In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, fixed
import numpy as np
from spectral import open_image

# ----------------------------
# 1️⃣ Load hyperspectral data
# ----------------------------
base = r"C:\Personel\Group Project\DataOps\T_G_test_5_closed_curtain_2025-09-24_12-39-55\capture"

dark = open_image(base + r"\DARKREF_T_G_test_5_closed_curtain_2025-09-24_12-39-55.hdr").load()
white = open_image(base + r"\WHITEREF_T_G_test_5_closed_curtain_2025-09-24_12-39-55.hdr").load()
img   = open_image(base + r"\T_G_test_5_closed_curtain_2025-09-24_12-39-55.hdr").load()

dark_ref = dark.mean(axis=0)
white_ref = white.mean(axis=0)

img_np = np.array(img)
dark_ref_np = np.array(dark_ref)
white_ref_np = np.array(white_ref)

epsilon = 1e-6
reflectance = np.clip((img_np - dark_ref_np) / (white_ref_np - dark_ref_np + epsilon), 0, 1)
H, W, B = reflectance.shape

# ----------------------------
# 2️⃣ Wavelengths & dynamic spectral groups
# ----------------------------
wavelengths = np.linspace(400, 1000, B)  # sensor wavelengths 400–1000 nm

bands_per_group = 10                  # desired bands per group
NUM_GROUPS = B // bands_per_group     # calculate total groups dynamically

groups = []
band_ranges = []
group_wavelength_ranges = []

for i in range(NUM_GROUPS):
    s = i * bands_per_group
    e = s + bands_per_group
    g = reflectance[:, :, s:e].mean(axis=2)
    g = (g - g.min()) / (g.max() - g.min() + 1e-8)
    groups.append(g)
    band_ranges.append((s, e-1))
    group_wavelength_ranges.append((wavelengths[s], wavelengths[e-1]))  # store nm range

# ----------------------------
# 3️⃣ RGB composite (optional)
# ----------------------------
R_band, G_band, B_band = 300, 200, 100
rgb_image = np.stack([
    reflectance[:, :, R_band],
    reflectance[:, :, G_band],
    reflectance[:, :, B_band]
], axis=2)

# ----------------------------
# 4️⃣ Visible region indices
# ----------------------------
visible_start_nm = 400
visible_end_nm = 700
visible_indices = np.where((wavelengths >= visible_start_nm) & (wavelengths <= visible_end_nm))[0]

# ----------------------------
# 5️⃣ Display function
# ----------------------------
def display_image_with_pixel(view_mode, pixel_x, pixel_y, group_index=0):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # --- Image display ---
    group_index = np.clip(group_index, 0, NUM_GROUPS-1)
    img_to_show = groups[group_index]
    wl_start, wl_end = group_wavelength_ranges[group_index]

    ax1.imshow(img_to_show, cmap='viridis')
    ax1.scatter(pixel_x, pixel_y, s=80, facecolors='none', edgecolors='red', linewidths=2)
    ax1.set_title(f"Spectral Group {group_index+1}/{NUM_GROUPS} | Wavelengths {wl_start:.1f}-{wl_end:.1f} nm")
    ax1.set_xlabel(f"Width (0 to {W-1})")
    ax1.set_ylabel(f"Height (0 to {H-1})")
    ax1.axis("on")

    # --- Spectral curve ---
    spectrum = reflectance[pixel_y, pixel_x, :]

    uv_end_nm = 400
    visible_mask = (wavelengths >= visible_start_nm) & (wavelengths <= visible_end_nm)
    nir_mask = wavelengths > visible_end_nm

    # UV region (0–400 nm)
    uv_wavelengths = np.linspace(0, uv_end_nm, 100)
    uv_spectrum = np.zeros_like(uv_wavelengths)
    ax2.plot(uv_wavelengths, uv_spectrum, color='violet', linewidth=1.5, linestyle='--', label='UV (0-400 nm) - No Data')

    # Visible (green) and NIR (red)
    ax2.plot(wavelengths[visible_mask], spectrum[visible_mask], 'g-', linewidth=1.5, label='Visible (400-700 nm)')
    ax2.plot(wavelengths[nir_mask], spectrum[nir_mask], 'r-', linewidth=1.5, label='NIR (700-1000 nm)')

    # Shaded regions
    ax2.axvspan(0, uv_end_nm, alpha=0.15, color='violet', label='_nolegend_')
    ax2.axvspan(visible_start_nm, visible_end_nm, alpha=0.1, color='green')
    ax2.axvspan(visible_end_nm, 1000, alpha=0.1, color='red')

    # Region labels
    ax2.text(200, 0.95, 'UV', fontsize=10, ha='center', color='purple', fontweight='bold')
    ax2.text(550, 0.95, 'Visible', fontsize=10, ha='center', color='green', fontweight='bold')
    ax2.text(850, 0.95, 'NIR', fontsize=10, ha='center', color='red', fontweight='bold')

    # X-axis ticks
    xticks = list(range(0, 1001, 50))
    ax2.set_xticks(xticks)
    ax2.tick_params(axis='x', labelsize=7, rotation=45)

    ax2.legend(loc='upper right', fontsize=8)
    ax2.set_xlabel("Wavelength (nm)")
    ax2.set_ylabel("Reflectance [0-1]")
    ax2.set_title(f"Spectral Curve at Pixel (X={pixel_x}, Y={pixel_y})")
    ax2.set_xlim(0, 1000)
    ax2.set_ylim(0, 1)
    ax2.grid(True, linewidth=0.3, alpha=0.5)

    plt.tight_layout()
    plt.show()


def display_image_with_box(view_mode, x_start, x_end, y_start, y_end, group_index=0):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # --- Image display ---
    group_index = np.clip(group_index, 0, NUM_GROUPS-1)
    img_to_show = groups[group_index]
    wl_start, wl_end = group_wavelength_ranges[group_index]

    ax1.imshow(img_to_show, cmap='viridis')
    # Draw rectangle
    rect_width = x_end - x_start
    rect_height = y_end - y_start
    ax1.add_patch(plt.Rectangle((x_start, y_start), rect_width, rect_height,
                                edgecolor='red', facecolor='none', linewidth=2))
    ax1.set_title(f"Spectral Group {group_index+1}/{NUM_GROUPS} | Wavelengths {wl_start:.1f}-{wl_end:.1f} nm")
    ax1.set_xlabel(f"Width (0 to {W-1})")
    ax1.set_ylabel(f"Height (0 to {H-1})")
    ax1.axis("on")

    # --- Spectral curve (average over box) ---
    x_start, x_end = sorted([x_start, x_end])
    y_start, y_end = sorted([y_start, y_end])
    selected_region = reflectance[y_start:y_end+1, x_start:x_end+1, :]
    spectrum = selected_region.mean(axis=(0,1))  # average over pixels

    uv_end_nm = 400
    visible_mask = (wavelengths >= visible_start_nm) & (wavelengths <= visible_end_nm)
    nir_mask = wavelengths > visible_end_nm

    # UV region (0–400 nm)
    uv_wavelengths = np.linspace(0, uv_end_nm, 100)
    uv_spectrum = np.zeros_like(uv_wavelengths)
    ax2.plot(uv_wavelengths, uv_spectrum, color='violet', linewidth=1.5, linestyle='--', label='UV (0-400 nm) - No Data')

    # Visible (green) and NIR (red)
    ax2.plot(wavelengths[visible_mask], spectrum[visible_mask], 'g-', linewidth=1.5, label='Visible (400-700 nm)')
    ax2.plot(wavelengths[nir_mask], spectrum[nir_mask], 'r-', linewidth=1.5, label='NIR (700-1000 nm)')

    # Shaded regions
    ax2.axvspan(0, uv_end_nm, alpha=0.15, color='violet', label='_nolegend_')
    ax2.axvspan(visible_start_nm, visible_end_nm, alpha=0.1, color='green')
    ax2.axvspan(visible_end_nm, 1000, alpha=0.1, color='red')

    # Region labels
    ax2.text(200, 0.95, 'UV', fontsize=10, ha='center', color='purple', fontweight='bold')
    ax2.text(550, 0.95, 'Visible', fontsize=10, ha='center', color='green', fontweight='bold')
    ax2.text(850, 0.95, 'NIR', fontsize=10, ha='center', color='red', fontweight='bold')

    # X-axis ticks
    xticks = list(range(0, 1001, 50))
    ax2.set_xticks(xticks)
    ax2.tick_params(axis='x', labelsize=7, rotation=45)

    ax2.legend(loc='upper right', fontsize=8)
    ax2.set_xlabel("Wavelength (nm)")
    ax2.set_ylabel("Reflectance [0-1]")
    ax2.set_title(f"Average Spectral Curve in Box ({x_start}-{x_end}, {y_start}-{y_end})")
    ax2.set_xlim(0, 1000)
    ax2.set_ylim(0, 1)
    ax2.grid(True, linewidth=0.3, alpha=0.5)

    plt.tight_layout()
    plt.show()

# ----------------------------
# 6️⃣ Interactive wrapper (Group only)
# ----------------------------
interact(display_image_with_pixel,
         view_mode=fixed("Group"),
         pixel_x=IntSlider(min=0, max=W-1, step=1, value=W//2, description='X'),
         pixel_y=IntSlider(min=0, max=H-1, step=1, value=H//2, description='Y'),
         group_index=IntSlider(min=0, max=NUM_GROUPS-1, step=1, value=0, description='Group'))

interact(display_image_with_box,
         view_mode=fixed("Group"),
         x_start=IntSlider(min=0, max=W-2, step=1, value=W//2-5, description='X Start'),
         x_end=IntSlider(min=1, max=W-1, step=1, value=W//2+5, description='X End'),
         y_start=IntSlider(min=0, max=H-2, step=1, value=H//2-5, description='Y Start'),
         y_end=IntSlider(min=1, max=H-1, step=1, value=H//2+5, description='Y End'),
         group_index=IntSlider(min=0, max=NUM_GROUPS-1, step=1, value=0, description='Group'))
